In [1]:
import os
import subprocess
import sys
from pathlib import Path
import re

REPO_URL = "https://github.com/MuhammadQaiser1921/swin-model.git"
REPO_NAME = "swin-model"
REPO_BRANCH = "AF_V3"
REPO_PATH = f"/kaggle/working/{REPO_NAME}"

print("Preparing repo branch:", REPO_BRANCH)

Preparing repo branch: AF_V3


In [2]:
if not os.path.exists(REPO_PATH):
    print(f"Cloning {REPO_BRANCH} from {REPO_URL}...")
    subprocess.run(["git", "clone", "-b", REPO_BRANCH, REPO_URL, REPO_PATH], check=True)
else:
    print(f"Repository exists. Updating branch {REPO_BRANCH}...")
    os.chdir(REPO_PATH)
    subprocess.run(["git", "fetch", "--all"], check=True)
    subprocess.run(["git", "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "pull", "origin", REPO_BRANCH], check=True)
    os.chdir("/kaggle/working")

sys.path.append(f"{REPO_PATH}/src")

req_file = f"{REPO_PATH}/requirements.txt"
if os.path.exists(req_file):
    print("Installing requirements...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req_file], check=True)

print("Repository and environment are ready.")

Cloning AF_V3 from https://github.com/MuhammadQaiser1921/swin-model.git...


Cloning into '/kaggle/working/swin-model'...


Installing requirements...
Repository and environment are ready.


In [3]:
transformer_path = Path(REPO_PATH) / "src" / "swin_transformer.py"
txt = transformer_path.read_text(encoding="utf-8")

# Force MLP activation to Mish in case branch still has GELU or Swish.
txt2 = txt.replace("activation='gelu'", "activation='mish'")
txt2 = txt2.replace("activation='swish'", "activation='mish'")

if txt2 != txt:
    transformer_path.write_text(txt2, encoding="utf-8")
    print("Updated MLP activation to Mish in swin_transformer.py")
else:
    print("No GELU/Swish pattern found to replace; file kept as-is.")

m = re.search(r"layers\.Dense\(int\(dim \* mlp_ratio\), activation='([^']+)'\)", txt2)
print("MLP activation in file:", m.group(1) if m else "NOT FOUND")
print("Contains attention softmax:", "tf.nn.softmax" in txt2)
print("Contains output sigmoid:", "activation='sigmoid'" in txt2)
print("Patched file:", transformer_path)

Updated MLP activation to Mish in swin_transformer.py
MLP activation in file: mish
Contains attention softmax: True
Contains output sigmoid: True
Patched file: /kaggle/working/swin-model/src/swin_transformer.py


In [4]:
import tensorflow as tf

physical_devices = tf.config.list_physical_devices('GPU')
print("TF Version:", tf.__version__)
print("GPUs:", physical_devices)

if physical_devices:
    for gpu in physical_devices:
        tf.config.experimental.set_memory_growth(gpu, True)
    print("GPU detected and configured.")
else:
    raise RuntimeError("No GPU found. Set Kaggle accelerator to T4.")

2026-04-06 11:50:55.028448: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775476255.209676      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775476255.261139      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775476255.687891      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775476255.687935      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775476255.687937      55 computation_placer.cc:177] computation placer alr

TF Version: 2.19.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU detected and configured.


In [5]:
from train_audio import Config

print("DATA_ROOT:", Config.DATA_ROOT)
print("TRAIN_DIR exists:", os.path.exists(Config.TRAIN_DIR))
print("VAL_DIR exists:", os.path.exists(Config.VAL_DIR))
print("TEST_DIR exists:", os.path.exists(Config.TEST_DIR))

if not os.path.exists(Config.DATA_ROOT):
    raise FileNotFoundError("Dataset not attached. Add bishertello/asvspoof-21-df-cqt first.")

DATA_ROOT: /kaggle/input/datasets/bishertello/asvspoof-21-df-cqt/my_dataset
TRAIN_DIR exists: True
VAL_DIR exists: True
TEST_DIR exists: True


In [6]:
from train_audio import load_and_prepare_data

data = load_and_prepare_data()

print("\nAudio Data Preparation Complete")
print(f"Training samples: {len(data['train_paths'])}")
print(f"Validation samples: {len(data['val_paths'])}")
print(f"Test samples: {len(data['test_paths'])}")

📂 Loading audio CQT dataset paths...
Train samples: 59325
Val samples: 18576
Test samples: 533927

Audio Data Preparation Complete
Training samples: 59325
Validation samples: 18576
Test samples: 533927


In [11]:
import importlib
import swin_transformer
import train_audio

importlib.reload(swin_transformer)
importlib.reload(train_audio)

from train_audio import run_training_session, Config

model, history, test_metrics = run_training_session(
    data,
    epochs=Config.epochs,
    batch_size=Config.batch_size,
    lr=Config.lr
)

print("\nAF_V3 audio training completed.")
print("Test metrics:", test_metrics)

🚀 Training audio model for 3 epochs...
Epoch 1/3
3708/3708 ━━━━━━━━━━━━━━━━━━━━ 0s 119ms/step - accuracy: 0.8982 - loss: 0.2808
💾 Saved best .pth at epoch 1 (val_accuracy: 0.8934) -> /kaggle/working/models/audio_checkpoints/audio_best_model_20260406_130138.pth
3708/3708 ━━━━━━━━━━━━━━━━━━━━ 514s 128ms/step - accuracy: 0.8982 - loss: 0.2808 - val_accuracy: 0.8934 - val_loss: 1.6369
Epoch 2/3
3708/3708 ━━━━━━━━━━━━━━━━━━━━ 453s 122ms/step - accuracy: 0.9420 - loss: 0.1441 - val_accuracy: 0.8934 - val_loss: 2.2557
Epoch 3/3
3708/3708 ━━━━━━━━━━━━━━━━━━━━ 445s 120ms/step - accuracy: 0.9529 - loss: 0.1360 - val_accuracy: 0.8934 - val_loss: 1.9252
🧪 Evaluating on test split...
33371/33371 ━━━━━━━━━━━━━━━━━━━━ 897s 27ms/step - accuracy: 0.9033 - loss: 0.8821
Test metrics: {'accuracy': 0.9772103428840637, 'loss': 0.197870671749115}

📌 Threshold metrics @ 0.50
Confusion Matrix [[TN, FP], [FN, TP]]:
[[  3848  11021]
 [  1147 517911]]
Precision: 0.9792 | Recall: 0.9978 | F1: 0.9884 | Accuracy: 0.

In [13]:
import json

results = {
    "branch": "AF_V3",
    "activation": {
        "mlp": "mish",
        "attention": "softmax",
        "output": "sigmoid"
    },
    "final_train_accuracy": float(history.history['accuracy'][-1]),
    "final_train_loss": float(history.history['loss'][-1]),
    "final_val_accuracy": float(history.history['val_accuracy'][-1]),
    "final_val_loss": float(history.history['val_loss'][-1]),
    "test_metrics": test_metrics
}

out_file = "/kaggle/working/af_v3_audio_metrics.json"
with open(out_file, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, default=str)

print("Saved metrics to:", out_file)
print(json.dumps(results, indent=2, default=str))

Saved metrics to: /kaggle/working/af_v3_audio_metrics.json
{
  "branch": "AF_V3",
  "activation": {
    "mlp": "mish",
    "attention": "softmax",
    "output": "sigmoid"
  },
  "final_train_accuracy": 0.9775474071502686,
  "final_train_loss": 0.060542624443769455,
  "final_val_accuracy": 0.893410861492157,
  "final_val_loss": 1.925218105316162,
  "test_metrics": {
    "accuracy": 0.9772103428840637,
    "loss": 0.197870671749115,
    "threshold_metrics": {
      "threshold": 0.5,
      "tn": 3848,
      "fp": 11021,
      "fn": 1147,
      "tp": 517911,
      "precision": 0.9791636732131733,
      "recall": 0.9977902276816657,
      "f1": 0.9883891971874688,
      "threshold_accuracy": 0.9772103677094435
    }
  }
}


In [8]:
import json

results = {
    "branch": "AF_V3",
    "activation": {
        "mlp": "mish",
        "attention": "softmax",
        "output": "sigmoid"
    },
    "final_train_accuracy": float(history.history['accuracy'][-1]),
    "final_train_loss": float(history.history['loss'][-1]),
    "final_val_accuracy": float(history.history['val_accuracy'][-1]),
    "final_val_loss": float(history.history['val_loss'][-1]),
    "test_metrics": test_metrics
}

out_file = "/kaggle/working/af_v3_audio_metrics.json"
with open(out_file, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, default=str)

print("Saved metrics to:", out_file)
print(json.dumps(results, indent=2, default=str))

NameError: name 'history' is not defined